# Kaggle — Universal Training Notebook (PConv / Vanilla / Gated)

**Purpose:** Train any of the three architectures on Kaggle P100,
reading the WikiArt dataset from the attached Kaggle dataset
`orivalo/processed` (contains `processed.zip` + `train/val/test.csv`).

**To use:**
1. Settings → Accelerator: **GPU P100**
2. Settings → Internet: **On**
3. Add data → search `processed` (your dataset) → Add
4. Edit `ARCH` in Cell 1 below to one of: `pconv_unet` | `unet_baseline` | `gated_unet`
5. Run all cells

**Resume across sessions on Kaggle:**
Kaggle's `/kaggle/working/` persists only when you **commit / Save
Version**.  When a session is about to end (~9 h limit), click
**Save Version → Quick Save** at the top right — this commits the
checkpoints inside `/kaggle/working/checkpoints/`.  In the next
session, add the previous notebook version as input data; the
Trainer's auto-resume picks up `last.pth` automatically.

Cells 1-7 execute independently per session.

## Cell 1 — Architecture Selection + Setup

Pick `ARCH`, install dependencies, clone source code from GitHub.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  Pick which architecture to train.
#  Same `configs/experiment_configs/<ARCH>.yaml` recipe is used across all 3,
#  so the only honest comparison axis is `model.arch`.
# ════════════════════════════════════════════════════════════════════════════
ARCH = 'pconv_unet'    # ← change to 'unet_baseline' or 'gated_unet'

assert ARCH in ('pconv_unet', 'unet_baseline', 'gated_unet'), \
    f'Unknown arch: {ARCH}'

# ── Install dependencies (Kaggle has torch + torchvision pre-installed) ─────
!pip install -q albumentations 'torchmetrics[image]' lpips pyyaml

# ── Clone source code from GitHub ───────────────────────────────────────────
import os, sys
from pathlib import Path

PROJECT_DIR = Path('/kaggle/working/art-restoration')

if not (PROJECT_DIR / 'src').exists():
    print('Cloning private repo from GitHub...')
    print('  Generate token at https://github.com/settings/tokens (scope: repo)')
    import getpass
    token = os.environ.get('GITHUB_TOKEN') or getpass.getpass('GitHub token: ')
    repo_url = f'https://{token}@github.com/orivalo/art_restoration.git'
    !git clone -b phase2-partial-conv {repo_url} {PROJECT_DIR}
    if not (PROJECT_DIR / 'src').exists():
        raise RuntimeError('Clone failed — check your token has `repo` scope.')
else:
    print(f'Project already cloned at {PROJECT_DIR}')
    !cd {PROJECT_DIR} && git pull

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

!cd {PROJECT_DIR} && git log --oneline -1

print(f'\nARCH         : {ARCH}')
print(f'PROJECT_DIR  : {PROJECT_DIR}')
print('Setup complete.')

## Cell 2 — GPU Verification

Hard-fail if no GPU.  Expect Tesla P100 (16 GB).

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU. Settings → Accelerator → GPU P100, then restart kernel.'
    )
props = torch.cuda.get_device_properties(0)
print(f'PyTorch  : {torch.__version__}')
print(f'GPU      : {props.name}')
print(f'VRAM     : {props.total_memory / 1024**3:.1f} GB')

## Cell 3 — Unzip dataset + rewrite splits to local paths

Reads `/kaggle/input/processed/processed.zip` and unzips into
`/kaggle/working/dataset_local/processed/`.  CSV files are read
from `/kaggle/input/processed/` and rewritten so paths point at
the local copy.

Idempotent — skips if already unzipped (e.g. resumed session).

In [ ]:
import shutil, time, zipfile
import pandas as pd

# ── Auto-detect Kaggle dataset mount path (legacy or new layout) ────────────
_CANDIDATES = [
    Path('/kaggle/input/processed'),
    Path('/kaggle/input/datasets/orivalo/processed'),
]
KAGGLE_INPUT = next((p for p in _CANDIDATES if p.exists()), None)
if KAGGLE_INPUT is None:
    for root in Path('/kaggle/input').rglob('train.csv'):
        KAGGLE_INPUT = root.parent
        break
if KAGGLE_INPUT is None:
    raise FileNotFoundError('Add Data → orivalo/processed first')
print(f'KAGGLE_INPUT : {KAGGLE_INPUT}')

LOCAL_DATA_ROOT = Path('/kaggle/working/dataset_local')
LOCAL_SPLITS    = Path('/kaggle/working/local_splits')

SOURCE_ZIP = KAGGLE_INPUT / 'processed.zip'

# ── Find the directory that actually contains the style sub-folders ─────────
# Kaggle's auto-extract + the user's local zip layout can produce extra
# nesting (processed/processed/processed/...).  Walk down from each candidate
# until we find a directory whose children look like style folders
# (impressionism, baroque, renaissance, post_impressionism, realism).
EXPECTED_STYLES = {'impressionism', 'baroque', 'renaissance', 'post_impressionism', 'realism'}

def find_styles_root(start: Path, max_depth: int = 5) -> Path | None:
    """Walk down from `start` looking for the level that holds the style folders."""
    if not start.exists():
        return None
    queue = [(start, 0)]
    while queue:
        d, depth = queue.pop(0)
        if depth > max_depth:
            continue
        try:
            children = {c.name for c in d.iterdir() if c.is_dir()}
        except (PermissionError, NotADirectoryError):
            continue
        if children & EXPECTED_STYLES:
            return d
        for c in d.iterdir():
            if c.is_dir():
                queue.append((c, depth + 1))
    return None

LOCAL_PROCESSED_STR = None

# Try the pre-extracted folder first
candidate = KAGGLE_INPUT / 'processed'
if candidate.exists():
    found = find_styles_root(candidate)
    if found is not None:
        LOCAL_PROCESSED_STR = str(found)
        print(f'Using pre-extracted dataset at {found}')

# Fall back to unzipping if the styles weren't found above
if LOCAL_PROCESSED_STR is None and SOURCE_ZIP.exists():
    if not (LOCAL_DATA_ROOT / 'processed').exists():
        print(f'Unzipping {SOURCE_ZIP} → {LOCAL_DATA_ROOT}/processed/')
        LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        with zipfile.ZipFile(SOURCE_ZIP, 'r') as zf:
            zf.extractall(LOCAL_DATA_ROOT / 'processed')
        print(f'  Unzipped in {(time.time() - t0) / 60:.1f} min')
    found = find_styles_root(LOCAL_DATA_ROOT / 'processed')
    if found is not None:
        LOCAL_PROCESSED_STR = str(found)

if LOCAL_PROCESSED_STR is None:
    raise FileNotFoundError(
        f'Could not locate style sub-folders ({EXPECTED_STYLES}) anywhere '
        f'under {KAGGLE_INPUT}.  Check the dataset structure manually.'
    )

n_local = sum(1 for _ in Path(LOCAL_PROCESSED_STR).rglob('*.jpg'))
print(f'  Local image count: {n_local:,}')

# ── Rewrite splits CSVs to local Kaggle paths ──────────────────────────────
LOCAL_SPLITS.mkdir(parents=True, exist_ok=True)
DRIVE_PREFIX = '/content/drive/MyDrive/art_restoration/data/processed'

print(f'\nRewriting splits CSVs → {LOCAL_SPLITS}/')
for csv_name in ['train.csv', 'val.csv', 'test.csv']:
    src = KAGGLE_INPUT / csv_name
    dst = LOCAL_SPLITS / csv_name
    df = pd.read_csv(src)
    df['path'] = df['path'].str.replace(DRIVE_PREFIX, LOCAL_PROCESSED_STR, regex=False)
    df.to_csv(dst, index=False)
    n_exist = df['path'].apply(lambda p: Path(p).exists()).sum()
    status = '✓' if n_exist == len(df) else f'⚠ {len(df) - n_exist} missing'
    print(f'  {csv_name:10s} : {len(df):>6,} rows   {n_exist:>6,} verified  {status}')

print('\nLocal data ready.')

## Cell 4 — Smoke test: build all 3 architectures

Verifies the registry, fp32 loss wrap, and forward pass on a tiny
synthetic batch.  Run once per session — fast (a few seconds).

In [ ]:
from src.models.registry import build_model, SUPPORTED_ARCHS

print('Smoke test: build_model dispatch')
print('━' * 60)
device = torch.device('cuda')
B, C, H, W = 1, 3, 256, 256
img  = torch.randn(B, C, H, W, device=device)
mask = (torch.rand(B, 1, H, W, device=device) > 0.4).float()

for arch in SUPPORTED_ARCHS:
    cfg = {'model': {'arch': arch, 'in_channels': 3, 'out_channels': 3, 'verbose': False}}
    model = build_model(cfg).to(device)
    n = sum(p.numel() for p in model.parameters())
    with torch.no_grad():
        output, ret_mask = model(img * mask, mask)
    assert output.shape == (B, C, H, W)
    assert ret_mask.shape == (B, 1, H, W)
    print(f'  {arch:14s} → {n:>11,} params  forward OK ✓')
    del model
    torch.cuda.empty_cache()
print('\nAll 3 architectures build correctly.')

## Cell 5 — Config

Load `configs/experiment_configs/<ARCH>.yaml`, override paths to use
Kaggle's local SSD for data and `/kaggle/working/` for checkpoints + logs.

**Per-platform tuning:** Kaggle P100 fits `batch_size: 8` comfortably
(checked via the smoke test) — overridden below.

In [ ]:
import yaml

CONFIG_PATH       = PROJECT_DIR / 'configs' / 'experiment_configs' / f'{ARCH}.yaml'
LOCAL_CONFIG_PATH = Path(f'/kaggle/working/{ARCH}.local.yaml')

with open(CONFIG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

# ── Kaggle-specific paths ───────────────────────────────────────────────────
cfg['data']['splits_dir'] = str(LOCAL_SPLITS)
cfg['checkpoint']['dir']  = f'/kaggle/working/checkpoints/{ARCH}'
cfg['logging']['log_dir'] = f'/kaggle/working/logs/{ARCH}'

# ── Kaggle P100 can fit a larger batch than T4 ──────────────────────────────
cfg['train']['batch_size'] = 8
cfg['data']['num_workers'] = 4

with open(LOCAL_CONFIG_PATH, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('Effective configuration')
print('━' * 60)
print(f"Experiment      : {cfg['experiment']['name']}")
print(f"Architecture    : {cfg['model']['arch']}")
print(f"Splits dir      : {cfg['data']['splits_dir']}")
print(f"Checkpoint dir  : {cfg['checkpoint']['dir']}")
print(f"Log dir         : {cfg['logging']['log_dir']}")
print()
print(f"Batch size      : {cfg['train']['batch_size']}  (P100)")
print(f"Num epochs      : {cfg['train']['num_epochs']}")
print(f"Learning rate   : {cfg['optim']['lr']}")
print(f"Mixed precision : {cfg['train']['amp']}")
print(f"Mask difficulty : {cfg['data']['difficulty']}")
print()
print(f'Wrote patched config → {LOCAL_CONFIG_PATH}')

## Cell 6 — Initialize Trainer

Builds model + loss + DataLoaders, attempts to resume from
`/kaggle/working/checkpoints/<ARCH>/last.pth` if it exists (carried
over from a previous notebook version).

In [ ]:
from src.training.trainer import Trainer
from src.utils.checkpoint import find_latest_checkpoint

ckpt_dir = Path(cfg['checkpoint']['dir'])
latest = find_latest_checkpoint(ckpt_dir) if ckpt_dir.exists() else None
if latest is not None:
    print(f'Found checkpoint : {latest.name} → will resume')
else:
    print('No checkpoint found — fresh training from scratch.')
print()

trainer = Trainer(LOCAL_CONFIG_PATH)

## Cell 7 — Train

Run the full multi-epoch loop.  Each epoch saves `last.pth` and
(if PSNR improved) `best.pth` to `/kaggle/working/checkpoints/<ARCH>/`.
Periodic snapshots `epoch_NNN.pth` every 5 epochs.

Expected per-epoch time on P100, batch=8: **~30-40 min**.  12 epochs ≈ **6-8 h**.
Fits in one 9 h Kaggle session.

**When session is about to end** (Kaggle warns ~30 min before timeout):
click **Save Version → Quick Save** at top right.  Next session, add
this notebook's previous output as data and resume from there.

In [ ]:
trainer.train()

## Cell 8 — Results

Plot training curves from `training_log.csv` and show the latest
validation visualisation grid.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

log_dir  = Path(cfg['logging']['log_dir'])
csv_path = log_dir / cfg['logging']['csv_filename']
fig_dir  = log_dir / 'figures'

if not csv_path.exists():
    print(f'No training log yet at {csv_path}')
else:
    df = pd.read_csv(csv_path)
    print(f'Loaded {len(df):,} epochs from {csv_path}')

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes[0, 0].plot(df['epoch'], df['train_total'], 'tab:blue', linewidth=2)
    axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Total training loss'); axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(df['epoch'], df['val_psnr'], 'tab:green', linewidth=2)
    axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('PSNR (dB)')
    axes[0, 1].set_title('Validation PSNR'); axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(df['epoch'], df['val_ssim'], 'tab:purple', linewidth=2)
    axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('SSIM')
    axes[1, 0].set_title('Validation SSIM'); axes[1, 0].grid(alpha=0.3)

    component_cols = [c for c in
        ['train_l1_valid', 'train_l1_hole', 'train_perceptual',
         'train_style', 'train_tv'] if c in df.columns]
    for col in component_cols:
        axes[1, 1].plot(df['epoch'], df[col], label=col.replace('train_', ''), alpha=0.85)
    axes[1, 1].set_yscale('log'); axes[1, 1].set_title('Loss components')
    axes[1, 1].grid(alpha=0.3, which='both'); axes[1, 1].legend(fontsize=8)

    plt.tight_layout(); plt.show()

    last = df.iloc[-1]
    best = df.iloc[df['val_psnr'].idxmax()]
    print(f"\nLast epoch: {int(last['epoch']):3d}  PSNR={last['val_psnr']:.2f} dB  SSIM={last['val_ssim']:.4f}")
    print(f"Best epoch: {int(best['epoch']):3d}  PSNR={best['val_psnr']:.2f} dB  SSIM={best['val_ssim']:.4f}")

if fig_dir.exists():
    figs = sorted(fig_dir.glob('val_epoch_*.png'))
    if figs:
        print(f'\nLast validation figure: {figs[-1].name}')
        img = Image.open(figs[-1])
        plt.figure(figsize=(14, 14))
        plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()

## After training finishes

1. **Click 'Save Version → Quick Save'** at top right.  This commits
   the entire `/kaggle/working/` directory (including `checkpoints/`)
   as a Kaggle notebook output.
2. To get checkpoints to Drive (for evaluation in `05_evaluate.ipynb`
   on Colab):
   - Open the saved notebook output → Output tab → download
     `checkpoints/<ARCH>/best.pth` (~100-200 MB)
   - Upload to Drive at `/MyDrive/art_restoration/checkpoints/<ARCH>/best.pth`
3. Repeat with a different `ARCH` value in Cell 1 to train the next model.

**Tip:** if you want to train all three on Kaggle, run this notebook
three times (once per `ARCH` value) — checkpoint folders are isolated
per architecture, so they don't conflict.